# Verification Thresholds and Operating Points

This notebook converts writer-verification similarity scores into reproducible decision thresholds and operating-point metrics.

## Purpose

- Load cached validation and official-test verification scores
- Select thresholds using writer-disjoint validation data only
- Apply frozen validation thresholds to official-test pairs
- Measure false acceptance rate, false rejection rate, true acceptance rate, and balanced accuracy
- Evaluate verification performance at fixed FAR operating points
- Compare global and condition-specific threshold behavior
- Avoid using raw accuracy as the primary metric because genuine and impostor pairs are highly imbalanced

## Experimental rule

All threshold selection is performed using the validation split.

Official-test labels are used only to measure performance after a validation-derived threshold has been fixed. Official-test results are not used to choose thresholds, models, augmentations, or hyperparameters.

Because the QUWI official-test split has already been inspected during baseline analysis, it should be treated as an internal secondary test set rather than a completely untouched final confirmatory set. Final publication-level confirmation should therefore include an external writer-disjoint dataset.

In [2]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import (
    balanced_accuracy_score,
    roc_auc_score,
    roc_curve,
)

In [3]:
PROJECT_ROOT = Path.cwd().resolve()

while PROJECT_ROOT != PROJECT_ROOT.parent and not (
    PROJECT_ROOT / "pyproject.toml"
).exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

assert (PROJECT_ROOT / "pyproject.toml").exists()

BASELINE_REPORT_DIR = (
    PROJECT_ROOT
    / "reports"
    / "writer_disjoint_baseline_verification"
)

CACHE_DIR = (
    BASELINE_REPORT_DIR
    / "cache"
)

VALIDATION_SCORE_PATH = (
    CACHE_DIR
    / "validation_verification_scores.csv.gz"
)

TEST_SCORE_PATH = (
    CACHE_DIR
    / "official_test_verification_scores.csv.gz"
)

REPORT_DIR = (
    PROJECT_ROOT
    / "reports"
    / "verification_thresholds"
)

REPORT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

validation_scores_df = pd.read_csv(
    VALIDATION_SCORE_PATH
)

test_scores_df = pd.read_csv(
    TEST_SCORE_PATH
)

score_columns = {
    "imagenet": "imagenet_cosine_score",
    "layer4_finetuned": "finetuned_cosine_score",
}

print(
    "Validation score file exists:",
    VALIDATION_SCORE_PATH.exists(),
)

print(
    "Official-test score file exists:",
    TEST_SCORE_PATH.exists(),
)

print(
    "Validation pairs:",
    len(validation_scores_df),
)

print(
    "Official-test pairs:",
    len(test_scores_df),
)

print(
    "Validation conditions:",
    validation_scores_df[
        "condition_id"
    ].nunique(),
)

print(
    "Official-test conditions:",
    test_scores_df[
        "condition_id"
    ].nunique(),
)

print(
    "Validation labels:",
    validation_scores_df[
        "pair_label"
    ].value_counts().sort_index().to_dict(),
)

print(
    "Official-test labels:",
    test_scores_df[
        "pair_label"
    ].value_counts().sort_index().to_dict(),
)

print(
    "Validation scores finite:",
    np.isfinite(
        validation_scores_df[
            list(score_columns.values())
        ].to_numpy()
    ).all(),
)

print(
    "Official-test scores finite:",
    np.isfinite(
        test_scores_df[
            list(score_columns.values())
        ].to_numpy()
    ).all(),
)

assert len(validation_scores_df) == 18_816
assert len(test_scores_df) == 223_494
assert validation_scores_df["condition_id"].nunique() == 6
assert test_scores_df["condition_id"].nunique() == 6

Validation score file exists: True
Official-test score file exists: True
Validation pairs: 18816
Official-test pairs: 223494
Validation conditions: 6
Official-test conditions: 6
Validation labels: {0: 18480, 1: 336}
Official-test labels: {0: 222336, 1: 1158}
Validation scores finite: True
Official-test scores finite: True


In [4]:
def metrics_at_threshold(
    labels,
    scores,
    threshold,
):
    labels = np.asarray(
        labels,
        dtype=np.int64,
    )

    scores = np.asarray(
        scores,
        dtype=np.float64,
    )

    predictions = (
        scores >= threshold
    ).astype(np.int64)

    genuine_mask = labels == 1
    impostor_mask = labels == 0

    true_accepts = (
        predictions[genuine_mask] == 1
    ).sum()

    false_rejects = (
        predictions[genuine_mask] == 0
    ).sum()

    false_accepts = (
        predictions[impostor_mask] == 1
    ).sum()

    true_rejects = (
        predictions[impostor_mask] == 0
    ).sum()

    tar = (
        true_accepts
        / genuine_mask.sum()
    )

    frr = (
        false_rejects
        / genuine_mask.sum()
    )

    far = (
        false_accepts
        / impostor_mask.sum()
    )

    trr = (
        true_rejects
        / impostor_mask.sum()
    )

    balanced_accuracy = (
        tar + trr
    ) / 2

    return {
        "threshold": float(
            threshold
        ),
        "tar": float(tar),
        "frr": float(frr),
        "far": float(far),
        "trr": float(trr),
        "balanced_accuracy": float(
            balanced_accuracy
        ),
        "true_accepts": int(
            true_accepts
        ),
        "false_rejects": int(
            false_rejects
        ),
        "false_accepts": int(
            false_accepts
        ),
        "true_rejects": int(
            true_rejects
        ),
    }


def select_eer_threshold(
    labels,
    scores,
):
    labels = np.asarray(
        labels,
        dtype=np.int64,
    )

    scores = np.asarray(
        scores,
        dtype=np.float64,
    )

    fpr, tpr, thresholds = roc_curve(
        labels,
        scores,
        pos_label=1,
    )

    fnr = 1.0 - tpr

    index = np.nanargmin(
        np.abs(
            fpr - fnr
        )
    )

    threshold = thresholds[index]

    operating_metrics = metrics_at_threshold(
        labels,
        scores,
        threshold,
    )

    operating_metrics[
        "threshold_rule"
    ] = "validation_eer"

    operating_metrics[
        "roc_auc"
    ] = float(
        roc_auc_score(
            labels,
            scores,
        )
    )

    operating_metrics[
        "eer_approx"
    ] = float(
        (
            operating_metrics["far"]
            + operating_metrics["frr"]
        )
        / 2
    )

    return operating_metrics


validation_eer_rows = []

for model_name, score_column in score_columns.items():
    result = select_eer_threshold(
        validation_scores_df[
            "pair_label"
        ],
        validation_scores_df[
            score_column
        ],
    )

    validation_eer_rows.append(
        {
            "model": model_name,
            **result,
        }
    )

validation_eer_thresholds_df = pd.DataFrame(
    validation_eer_rows
)

display(
    validation_eer_thresholds_df[
        [
            "model",
            "roc_auc",
            "threshold",
            "far",
            "frr",
            "eer_approx",
            "tar",
            "balanced_accuracy",
        ]
    ].round(6)
)

assert len(
    validation_eer_thresholds_df
) == 2

assert validation_eer_thresholds_df[
    "threshold"
].between(-1, 1).all()

assert validation_eer_thresholds_df[
    "far"
].between(0, 1).all()

assert validation_eer_thresholds_df[
    "frr"
].between(0, 1).all()

,model,roc_auc,threshold,far,frr,eer_approx,tar,balanced_accuracy
0,imagenet,0.612379,0.939676,0.432684,0.431548,0.432116,0.568452,0.567884
1,layer4_finetuned,0.674850,0.925156,0.366721,0.366071,0.366396,0.633929,0.633604


In [5]:
eer_transfer_rows = []

for model_name, score_column in score_columns.items():
    threshold = (
        validation_eer_thresholds_df.loc[
            validation_eer_thresholds_df[
                "model"
            ] == model_name,
            "threshold",
        ]
        .iloc[0]
    )

    for split_name, score_df in {
        "validation": validation_scores_df,
        "official_test": test_scores_df,
    }.items():
        metrics = metrics_at_threshold(
            score_df["pair_label"],
            score_df[score_column],
            threshold,
        )

        eer_transfer_rows.append(
            {
                "model": model_name,
                "experiment_split": split_name,
                "threshold_source": "validation",
                "threshold_rule": "validation_eer",
                **metrics,
            }
        )

eer_threshold_transfer_df = pd.DataFrame(
    eer_transfer_rows
)

display(
    eer_threshold_transfer_df[
        [
            "model",
            "experiment_split",
            "threshold",
            "far",
            "frr",
            "tar",
            "trr",
            "balanced_accuracy",
        ]
    ].round(6)
)

for model_name in score_columns:
    model_rows = eer_threshold_transfer_df[
        eer_threshold_transfer_df[
            "model"
        ] == model_name
    ]

    validation_row = model_rows[
        model_rows[
            "experiment_split"
        ] == "validation"
    ].iloc[0]

    test_row = model_rows[
        model_rows[
            "experiment_split"
        ] == "official_test"
    ].iloc[0]

    print()
    print(
        model_name,
        "threshold:",
        round(
            validation_row["threshold"],
            6,
        ),
    )

    print(
        model_name,
        "validation FAR:",
        round(
            validation_row["far"] * 100,
            4,
        ),
        "%",
    )

    print(
        model_name,
        "test FAR:",
        round(
            test_row["far"] * 100,
            4,
        ),
        "%",
    )

    print(
        model_name,
        "validation FRR:",
        round(
            validation_row["frr"] * 100,
            4,
        ),
        "%",
    )

    print(
        model_name,
        "test FRR:",
        round(
            test_row["frr"] * 100,
            4,
        ),
        "%",
    )

    print(
        model_name,
        "test balanced accuracy:",
        round(
            test_row[
                "balanced_accuracy"
            ] * 100,
            4,
        ),
        "%",
    )

assert len(
    eer_threshold_transfer_df
) == 4

,model,experiment_split,threshold,far,frr,tar,trr,balanced_accuracy
0,imagenet,validation,0.939676,0.432684,0.431548,0.568452,0.567316,0.567884
1,imagenet,official_test,0.939676,0.400961,0.402418,0.597582,0.599039,0.598311
2,layer4_finetuned,validation,0.925156,0.366721,0.366071,0.633929,0.633279,0.633604
3,layer4_finetuned,official_test,0.925156,0.337278,0.361831,0.638169,0.662722,0.650446



imagenet threshold: 0.939676
imagenet validation FAR: 43.2684 %
imagenet test FAR: 40.0961 %
imagenet validation FRR: 43.1548 %
imagenet test FRR: 40.2418 %
imagenet test balanced accuracy: 59.8311 %

layer4_finetuned threshold: 0.925156
layer4_finetuned validation FAR: 36.6721 %
layer4_finetuned test FAR: 33.7278 %
layer4_finetuned validation FRR: 36.6071 %
layer4_finetuned test FRR: 36.1831 %
layer4_finetuned test balanced accuracy: 65.0446 %


In [6]:
def select_fixed_far_threshold(
    labels,
    scores,
    target_far,
):
    labels = np.asarray(
        labels,
        dtype=np.int64,
    )

    scores = np.asarray(
        scores,
        dtype=np.float64,
    )

    fpr, tpr, thresholds = roc_curve(
        labels,
        scores,
        pos_label=1,
    )

    valid_indices = np.where(
        fpr <= target_far
    )[0]

    assert len(valid_indices) > 0

    best_tpr = tpr[
        valid_indices
    ].max()

    candidate_indices = (
        valid_indices[
            tpr[valid_indices]
            == best_tpr
        ]
    )

    selected_index = (
        candidate_indices[-1]
    )

    threshold = thresholds[
        selected_index
    ]

    metrics = metrics_at_threshold(
        labels,
        scores,
        threshold,
    )

    return {
        "target_far": float(
            target_far
        ),
        "threshold": float(
            threshold
        ),
        "achieved_far": float(
            metrics["far"]
        ),
        "tar": float(
            metrics["tar"]
        ),
        "frr": float(
            metrics["frr"]
        ),
        "balanced_accuracy": float(
            metrics[
                "balanced_accuracy"
            ]
        ),
    }


FAR_TARGETS = [
    0.10,
    0.05,
    0.01,
    0.001,
]

fixed_far_rows = []

for model_name, score_column in score_columns.items():
    for target_far in FAR_TARGETS:
        result = select_fixed_far_threshold(
            validation_scores_df[
                "pair_label"
            ],
            validation_scores_df[
                score_column
            ],
            target_far,
        )

        fixed_far_rows.append(
            {
                "model": model_name,
                "threshold_source": "validation",
                **result,
            }
        )

validation_fixed_far_thresholds_df = pd.DataFrame(
    fixed_far_rows
)

display(
    validation_fixed_far_thresholds_df.round(
        6
    )
)

for model_name in score_columns:
    print()
    print(
        model_name,
        "validation operating points"
    )

    display(
        validation_fixed_far_thresholds_df[
            validation_fixed_far_thresholds_df[
                "model"
            ] == model_name
        ][
            [
                "target_far",
                "threshold",
                "achieved_far",
                "tar",
                "frr",
                "balanced_accuracy",
            ]
        ].round(6)
    )

assert len(
    validation_fixed_far_thresholds_df
) == 8

assert (
    validation_fixed_far_thresholds_df[
        "achieved_far"
    ]
    <= validation_fixed_far_thresholds_df[
        "target_far"
    ]
    + 1e-12
).all()

,model,threshold_source,target_far,threshold,achieved_far,tar,frr,balanced_accuracy
0,imagenet,validation,0.100,0.964000,0.099297,0.217262,0.782738,0.558983
1,imagenet,validation,0.050,0.969878,0.049297,0.139881,0.860119,0.545292
2,imagenet,validation,0.010,0.978928,0.009253,0.032738,0.967262,0.511742
3,imagenet,validation,0.001,0.986227,0.000920,0.002976,0.997024,0.501028
4,layer4_finetuned,validation,0.100,0.949022,0.097998,0.324405,0.675595,0.613203
5,layer4_finetuned,validation,0.050,0.956082,0.048214,0.211310,0.788690,0.581548
6,layer4_finetuned,validation,0.010,0.966886,0.009957,0.077381,0.922619,0.533712
7,layer4_finetuned,validation,0.001,0.976120,0.000812,0.008929,0.991071,0.504058



imagenet validation operating points


,target_far,threshold,achieved_far,tar,frr,balanced_accuracy
0,0.100,0.964000,0.099297,0.217262,0.782738,0.558983
1,0.050,0.969878,0.049297,0.139881,0.860119,0.545292
2,0.010,0.978928,0.009253,0.032738,0.967262,0.511742
3,0.001,0.986227,0.000920,0.002976,0.997024,0.501028



layer4_finetuned validation operating points


,target_far,threshold,achieved_far,tar,frr,balanced_accuracy
4,0.100,0.949022,0.097998,0.324405,0.675595,0.613203
5,0.050,0.956082,0.048214,0.211310,0.788690,0.581548
6,0.010,0.966886,0.009957,0.077381,0.922619,0.533712
7,0.001,0.976120,0.000812,0.008929,0.991071,0.504058


In [7]:
fixed_far_transfer_rows = []

for row in validation_fixed_far_thresholds_df.itertuples(
    index=False
):
    model_name = row.model
    score_column = score_columns[
        model_name
    ]

    threshold = row.threshold
    target_far = row.target_far

    for split_name, score_df in {
        "validation": validation_scores_df,
        "official_test": test_scores_df,
    }.items():
        metrics = metrics_at_threshold(
            score_df["pair_label"],
            score_df[score_column],
            threshold,
        )

        fixed_far_transfer_rows.append(
            {
                "model": model_name,
                "experiment_split": split_name,
                "threshold_source": "validation",
                "target_far": target_far,
                "threshold": threshold,
                "far": metrics["far"],
                "tar": metrics["tar"],
                "frr": metrics["frr"],
                "trr": metrics["trr"],
                "balanced_accuracy": metrics[
                    "balanced_accuracy"
                ],
            }
        )

fixed_far_transfer_df = pd.DataFrame(
    fixed_far_transfer_rows
)

display(
    fixed_far_transfer_df.round(6)
)

assert len(
    fixed_far_transfer_df
) == 16

assert fixed_far_transfer_df[
    "threshold"
].between(-1, 1).all()

assert fixed_far_transfer_df[
    "far"
].between(0, 1).all()

assert fixed_far_transfer_df[
    "tar"
].between(0, 1).all()

,model,experiment_split,threshold_source,target_far,threshold,far,tar,frr,trr,balanced_accuracy
0,imagenet,validation,validation,0.100,0.964000,0.099297,0.217262,0.782738,0.900703,0.558983
1,imagenet,official_test,validation,0.100,0.964000,0.081615,0.189119,0.810881,0.918385,0.553752
2,imagenet,validation,validation,0.050,0.969878,0.049297,0.139881,0.860119,0.950703,0.545292
3,imagenet,official_test,validation,0.050,0.969878,0.039121,0.091537,0.908463,0.960879,0.526208
4,imagenet,validation,validation,0.010,0.978928,0.009253,0.032738,0.967262,0.990747,0.511742
5,imagenet,official_test,validation,0.010,0.978928,0.006904,0.018135,0.981865,0.993096,0.505615
6,imagenet,validation,validation,0.001,0.986227,0.000920,0.002976,0.997024,0.999080,0.501028
7,imagenet,official_test,validation,0.001,0.986227,0.000585,0.003454,0.996546,0.999415,0.501435
8,layer4_finetuned,validation,validation,0.100,0.949022,0.097998,0.324405,0.675595,0.902002,0.613203
9,layer4_finetuned,official_test,validation,0.100,0.949022,0.083081,0.287565,0.712435,0.916919,0.602242


In [8]:
fixed_far_comparison_df = (
    fixed_far_transfer_df
    .pivot(
        index=[
            "model",
            "target_far",
            "threshold",
        ],
        columns="experiment_split",
        values=[
            "far",
            "tar",
            "frr",
            "balanced_accuracy",
        ],
    )
)

fixed_far_comparison_df.columns = [
    f"{metric}_{split_name}"
    for metric, split_name
    in fixed_far_comparison_df.columns
]

fixed_far_comparison_df = (
    fixed_far_comparison_df
    .reset_index()
)

fixed_far_comparison_df[
    "far_shift"
] = (
    fixed_far_comparison_df[
        "far_official_test"
    ]
    - fixed_far_comparison_df[
        "far_validation"
    ]
)

fixed_far_comparison_df[
    "tar_shift"
] = (
    fixed_far_comparison_df[
        "tar_official_test"
    ]
    - fixed_far_comparison_df[
        "tar_validation"
    ]
)

fixed_far_comparison_df[
    "balanced_accuracy_shift"
] = (
    fixed_far_comparison_df[
        "balanced_accuracy_official_test"
    ]
    - fixed_far_comparison_df[
        "balanced_accuracy_validation"
    ]
)

display(
    fixed_far_comparison_df.round(6)
)

for model_name in score_columns:
    print()
    print(
        model_name,
        "threshold transfer"
    )

    display(
        fixed_far_comparison_df[
            fixed_far_comparison_df[
                "model"
            ] == model_name
        ][
            [
                "target_far",
                "threshold",
                "far_validation",
                "far_official_test",
                "far_shift",
                "tar_validation",
                "tar_official_test",
                "tar_shift",
            ]
        ].round(6)
    )

print(
    "Maximum absolute FAR shift:",
    round(
        fixed_far_comparison_df[
            "far_shift"
        ].abs().max(),
        6,
    ),
)

print(
    "Maximum absolute TAR shift:",
    round(
        fixed_far_comparison_df[
            "tar_shift"
        ].abs().max(),
        6,
    ),
)

,model,target_far,threshold,far_official_test,far_validation,tar_official_test,tar_validation,frr_official_test,frr_validation,balanced_accuracy_official_test,balanced_accuracy_validation,far_shift,tar_shift,balanced_accuracy_shift
0,imagenet,0.001,0.986227,0.000585,0.000920,0.003454,0.002976,0.996546,0.997024,0.501435,0.501028,-0.000335,0.000478,0.000407
1,imagenet,0.010,0.978928,0.006904,0.009253,0.018135,0.032738,0.981865,0.967262,0.505615,0.511742,-0.002349,-0.014603,-0.006127
2,imagenet,0.050,0.969878,0.039121,0.049297,0.091537,0.139881,0.908463,0.860119,0.526208,0.545292,-0.010176,-0.048344,-0.019084
3,imagenet,0.100,0.964000,0.081615,0.099297,0.189119,0.217262,0.810881,0.782738,0.553752,0.558983,-0.017681,-0.028143,-0.005231
4,layer4_finetuned,0.001,0.976120,0.000778,0.000812,0.009499,0.008929,0.990501,0.991071,0.504361,0.504058,-0.000034,0.000571,0.000302
5,layer4_finetuned,0.010,0.966886,0.007435,0.009957,0.045769,0.077381,0.954231,0.922619,0.519167,0.533712,-0.002522,-0.031612,-0.014545
6,layer4_finetuned,0.050,0.956082,0.039274,0.048214,0.170121,0.211310,0.829879,0.788690,0.565424,0.581548,-0.008940,-0.041189,-0.016124
7,layer4_finetuned,0.100,0.949022,0.083081,0.097998,0.287565,0.324405,0.712435,0.675595,0.602242,0.613203,-0.014916,-0.036840,-0.010962



imagenet threshold transfer


,target_far,threshold,far_validation,far_official_test,far_shift,tar_validation,tar_official_test,tar_shift
0,0.001,0.986227,0.000920,0.000585,-0.000335,0.002976,0.003454,0.000478
1,0.010,0.978928,0.009253,0.006904,-0.002349,0.032738,0.018135,-0.014603
2,0.050,0.969878,0.049297,0.039121,-0.010176,0.139881,0.091537,-0.048344
3,0.100,0.964000,0.099297,0.081615,-0.017681,0.217262,0.189119,-0.028143



layer4_finetuned threshold transfer


,target_far,threshold,far_validation,far_official_test,far_shift,tar_validation,tar_official_test,tar_shift
4,0.001,0.976120,0.000812,0.000778,-0.000034,0.008929,0.009499,0.000571
5,0.010,0.966886,0.009957,0.007435,-0.002522,0.077381,0.045769,-0.031612
6,0.050,0.956082,0.048214,0.039274,-0.008940,0.211310,0.170121,-0.041189
7,0.100,0.949022,0.097998,0.083081,-0.014916,0.324405,0.287565,-0.036840


Maximum absolute FAR shift: 0.017681
Maximum absolute TAR shift: 0.048344


In [9]:
global_threshold_condition_rows = []

for model_name, score_column in score_columns.items():
    threshold = (
        validation_eer_thresholds_df.loc[
            validation_eer_thresholds_df[
                "model"
            ] == model_name,
            "threshold",
        ]
        .iloc[0]
    )

    for split_name, score_df in {
        "validation": validation_scores_df,
        "official_test": test_scores_df,
    }.items():
        for condition_id, condition_df in score_df.groupby(
            "condition_id",
            sort=False,
        ):
            metrics = metrics_at_threshold(
                condition_df["pair_label"],
                condition_df[score_column],
                threshold,
            )

            global_threshold_condition_rows.append(
                {
                    "model": model_name,
                    "experiment_split": split_name,
                    "condition_id": condition_id,
                    "script_relation": condition_df[
                        "script_relation"
                    ].iloc[0],
                    "text_relation": condition_df[
                        "text_relation"
                    ].iloc[0],
                    "threshold_source": "validation_global_eer",
                    **metrics,
                }
            )

global_threshold_condition_df = pd.DataFrame(
    global_threshold_condition_rows
)

display(
    global_threshold_condition_df[
        [
            "model",
            "experiment_split",
            "condition_id",
            "threshold",
            "far",
            "frr",
            "tar",
            "balanced_accuracy",
        ]
    ].round(6)
)

finetuned_global_condition_df = (
    global_threshold_condition_df[
        global_threshold_condition_df[
            "model"
        ] == "layer4_finetuned"
    ]
)

for split_name in [
    "validation",
    "official_test",
]:
    rows = finetuned_global_condition_df[
        finetuned_global_condition_df[
            "experiment_split"
        ] == split_name
    ]

    print()
    print(
        split_name,
        "fine-tuned condition FAR range:",
        round(
            rows["far"].min() * 100,
            4,
        ),
        "to",
        round(
            rows["far"].max() * 100,
            4,
        ),
        "%",
    )

    print(
        split_name,
        "fine-tuned condition FRR range:",
        round(
            rows["frr"].min() * 100,
            4,
        ),
        "to",
        round(
            rows["frr"].max() * 100,
            4,
        ),
        "%",
    )

assert len(
    global_threshold_condition_df
) == 24

,model,experiment_split,condition_id,threshold,far,frr,tar,balanced_accuracy
0,imagenet,validation,arabic_variable_same,0.939676,0.455844,0.303571,0.696429,0.620292
1,imagenet,validation,english_variable_same,0.939676,0.359416,0.500000,0.500000,0.570292
2,imagenet,validation,cross_variable_variable,0.939676,0.700649,0.160714,0.839286,0.569318
3,imagenet,validation,cross_variable_same,0.939676,0.284416,0.660714,0.339286,0.527435
4,imagenet,validation,cross_same_variable,0.939676,0.322727,0.660714,0.339286,0.508279
5,imagenet,validation,cross_same_same,0.939676,0.473052,0.303571,0.696429,0.611688
6,imagenet,official_test,arabic_variable_same,0.939676,0.411701,0.316062,0.683938,0.636118
7,imagenet,official_test,english_variable_same,0.939676,0.349633,0.347150,0.652850,0.651608
8,imagenet,official_test,cross_variable_variable,0.939676,0.672496,0.191710,0.808290,0.567897
9,imagenet,official_test,cross_variable_same,0.939676,0.214702,0.715026,0.284974,0.535136



validation fine-tuned condition FAR range: 24.6753 to 53.539 %
validation fine-tuned condition FRR range: 23.2143 to 60.7143 %

official_test fine-tuned condition FAR range: 20.0345 to 49.1256 %
official_test fine-tuned condition FRR range: 21.2435 to 62.6943 %


In [10]:
condition_threshold_rows = []

for model_name, score_column in score_columns.items():
    for condition_id, condition_df in validation_scores_df.groupby(
        "condition_id",
        sort=False,
    ):
        result = select_eer_threshold(
            condition_df["pair_label"],
            condition_df[score_column],
        )

        condition_threshold_rows.append(
            {
                "model": model_name,
                "condition_id": condition_id,
                "script_relation": condition_df[
                    "script_relation"
                ].iloc[0],
                "text_relation": condition_df[
                    "text_relation"
                ].iloc[0],
                **result,
            }
        )

validation_condition_thresholds_df = pd.DataFrame(
    condition_threshold_rows
)

condition_threshold_transfer_rows = []

for row in validation_condition_thresholds_df.itertuples(
    index=False
):
    score_column = score_columns[
        row.model
    ]

    for split_name, score_df in {
        "validation": validation_scores_df,
        "official_test": test_scores_df,
    }.items():
        condition_df = score_df[
            score_df["condition_id"]
            == row.condition_id
        ]

        metrics = metrics_at_threshold(
            condition_df["pair_label"],
            condition_df[score_column],
            row.threshold,
        )

        condition_threshold_transfer_rows.append(
            {
                "model": row.model,
                "condition_id": row.condition_id,
                "experiment_split": split_name,
                "threshold_source": "validation_condition_eer",
                "threshold": row.threshold,
                "far": metrics["far"],
                "frr": metrics["frr"],
                "tar": metrics["tar"],
                "balanced_accuracy": metrics[
                    "balanced_accuracy"
                ],
            }
        )

condition_threshold_transfer_df = pd.DataFrame(
    condition_threshold_transfer_rows
)

display(
    validation_condition_thresholds_df[
        [
            "model",
            "condition_id",
            "threshold",
            "far",
            "frr",
            "tar",
            "balanced_accuracy",
        ]
    ].round(6)
)

print(
    "Validation condition thresholds:",
    len(
        validation_condition_thresholds_df
    ),
)

print(
    "Transferred condition evaluations:",
    len(
        condition_threshold_transfer_df
    ),
)

print()
print(
    "Fine-tuned condition-specific thresholds:"
)

display(
    validation_condition_thresholds_df[
        validation_condition_thresholds_df[
            "model"
        ] == "layer4_finetuned"
    ][
        [
            "condition_id",
            "threshold",
            "far",
            "frr",
            "tar",
        ]
    ].round(6)
)

assert len(
    validation_condition_thresholds_df
) == 12

assert len(
    condition_threshold_transfer_df
) == 24

,model,condition_id,threshold,far,frr,tar,balanced_accuracy
0,imagenet,arabic_variable_same,0.944537,0.378571,0.375000,0.625000,0.623214
1,imagenet,english_variable_same,0.934398,0.435065,0.428571,0.571429,0.568182
2,imagenet,cross_variable_variable,0.961077,0.377273,0.375000,0.625000,0.623864
3,imagenet,cross_variable_same,0.927260,0.482792,0.482143,0.517857,0.517532
4,imagenet,cross_same_variable,0.931418,0.436039,0.428571,0.571429,0.567695
5,imagenet,cross_same_same,0.945905,0.382468,0.375000,0.625000,0.621266
6,layer4_finetuned,arabic_variable_same,0.932528,0.299351,0.303571,0.696429,0.698539
7,layer4_finetuned,english_variable_same,0.927690,0.339935,0.339286,0.660714,0.660390
8,layer4_finetuned,cross_variable_variable,0.935592,0.393831,0.392857,0.607143,0.606656
9,layer4_finetuned,cross_variable_same,0.910292,0.446753,0.446429,0.553571,0.553409


Validation condition thresholds: 12
Transferred condition evaluations: 24

Fine-tuned condition-specific thresholds:


,condition_id,threshold,far,frr,tar
6,arabic_variable_same,0.932528,0.299351,0.303571,0.696429
7,english_variable_same,0.927690,0.339935,0.339286,0.660714
8,cross_variable_variable,0.935592,0.393831,0.392857,0.607143
9,cross_variable_same,0.910292,0.446753,0.446429,0.553571
10,cross_same_variable,0.914485,0.404870,0.410714,0.589286
11,cross_same_same,0.927901,0.337013,0.339286,0.660714


In [11]:
global_diagnostic_df = (
    global_threshold_condition_df[
        [
            "model",
            "experiment_split",
            "condition_id",
            "far",
            "frr",
            "tar",
            "balanced_accuracy",
        ]
    ]
    .copy()
)

global_diagnostic_df[
    "threshold_strategy"
] = "global_validation_eer"

condition_diagnostic_df = (
    condition_threshold_transfer_df[
        [
            "model",
            "experiment_split",
            "condition_id",
            "far",
            "frr",
            "tar",
            "balanced_accuracy",
        ]
    ]
    .copy()
)

condition_diagnostic_df[
    "threshold_strategy"
] = "condition_validation_eer"

threshold_diagnostic_df = pd.concat(
    [
        global_diagnostic_df,
        condition_diagnostic_df,
    ],
    ignore_index=True,
)

threshold_diagnostic_df[
    "absolute_far_frr_gap"
] = (
    threshold_diagnostic_df["far"]
    - threshold_diagnostic_df["frr"]
).abs()

threshold_diagnostic_summary_df = (
    threshold_diagnostic_df
    .groupby(
        [
            "model",
            "experiment_split",
            "threshold_strategy",
        ],
        as_index=False,
    )
    .agg(
        conditions=(
            "condition_id",
            "nunique",
        ),
        mean_far=(
            "far",
            "mean",
        ),
        mean_frr=(
            "frr",
            "mean",
        ),
        mean_tar=(
            "tar",
            "mean",
        ),
        mean_balanced_accuracy=(
            "balanced_accuracy",
            "mean",
        ),
        mean_absolute_far_frr_gap=(
            "absolute_far_frr_gap",
            "mean",
        ),
        max_absolute_far_frr_gap=(
            "absolute_far_frr_gap",
            "max",
        ),
    )
)

display(
    threshold_diagnostic_summary_df.round(
        6
    )
)

finetuned_diagnostic_summary_df = (
    threshold_diagnostic_summary_df[
        threshold_diagnostic_summary_df[
            "model"
        ] == "layer4_finetuned"
    ]
)

display(
    finetuned_diagnostic_summary_df[
        [
            "experiment_split",
            "threshold_strategy",
            "mean_far",
            "mean_frr",
            "mean_absolute_far_frr_gap",
            "max_absolute_far_frr_gap",
        ]
    ].round(6)
)

finetuned_thresholds = (
    validation_condition_thresholds_df[
        validation_condition_thresholds_df[
            "model"
        ] == "layer4_finetuned"
    ]["threshold"]
)

print(
    "Fine-tuned condition threshold minimum:",
    round(
        finetuned_thresholds.min(),
        6,
    ),
)

print(
    "Fine-tuned condition threshold maximum:",
    round(
        finetuned_thresholds.max(),
        6,
    ),
)

print(
    "Fine-tuned condition threshold spread:",
    round(
        finetuned_thresholds.max()
        - finetuned_thresholds.min(),
        6,
    ),
)

assert len(
    threshold_diagnostic_summary_df
) == 8

,model,experiment_split,threshold_strategy,conditions,mean_far,mean_frr,mean_tar,mean_balanced_accuracy,mean_absolute_far_frr_gap,max_absolute_far_frr_gap
0,imagenet,official_test,condition_validation_eer,6,0.380249,0.414508,0.585492,0.602622,0.096651,0.146832
1,imagenet,official_test,global_validation_eer,6,0.400961,0.402418,0.597582,0.598311,0.224210,0.500324
2,imagenet,validation,condition_validation_eer,6,0.415368,0.410714,0.589286,0.586959,0.004654,0.007468
3,imagenet,validation,global_validation_eer,6,0.432684,0.431548,0.568452,0.567884,0.286093,0.539935
4,layer4_finetuned,official_test,condition_validation_eer,6,0.340273,0.328152,0.671848,0.665787,0.042481,0.097960
5,layer4_finetuned,official_test,global_validation_eer,6,0.337278,0.361831,0.638169,0.650446,0.193909,0.426598
6,layer4_finetuned,validation,condition_validation_eer,6,0.370292,0.372024,0.627976,0.628842,0.002381,0.005844
7,layer4_finetuned,validation,global_validation_eer,6,0.366721,0.366071,0.633929,0.633604,0.204654,0.360390


,experiment_split,threshold_strategy,mean_far,mean_frr,mean_absolute_far_frr_gap,max_absolute_far_frr_gap
4,official_test,condition_validation_eer,0.340273,0.328152,0.042481,0.097960
5,official_test,global_validation_eer,0.337278,0.361831,0.193909,0.426598
6,validation,condition_validation_eer,0.370292,0.372024,0.002381,0.005844
7,validation,global_validation_eer,0.366721,0.366071,0.204654,0.360390


Fine-tuned condition threshold minimum: 0.910292
Fine-tuned condition threshold maximum: 0.935592
Fine-tuned condition threshold spread: 0.025299


In [12]:
EER_THRESHOLD_PATH = (
    REPORT_DIR
    / "validation_eer_thresholds.csv"
)

EER_TRANSFER_PATH = (
    REPORT_DIR
    / "eer_threshold_transfer.csv"
)

FIXED_FAR_THRESHOLD_PATH = (
    REPORT_DIR
    / "validation_fixed_far_thresholds.csv"
)

FIXED_FAR_TRANSFER_PATH = (
    REPORT_DIR
    / "fixed_far_threshold_transfer.csv"
)

FIXED_FAR_COMPARISON_PATH = (
    REPORT_DIR
    / "fixed_far_transfer_comparison.csv"
)

GLOBAL_CONDITION_PATH = (
    REPORT_DIR
    / "global_threshold_condition_metrics.csv"
)

CONDITION_THRESHOLD_PATH = (
    REPORT_DIR
    / "validation_condition_eer_thresholds.csv"
)

CONDITION_TRANSFER_PATH = (
    REPORT_DIR
    / "condition_threshold_transfer.csv"
)

THRESHOLD_DIAGNOSTIC_PATH = (
    REPORT_DIR
    / "threshold_calibration_diagnostic.csv"
)

SUMMARY_PATH = (
    REPORT_DIR
    / "verification_threshold_summary.json"
)

validation_eer_thresholds_df.to_csv(
    EER_THRESHOLD_PATH,
    index=False,
)

eer_threshold_transfer_df.to_csv(
    EER_TRANSFER_PATH,
    index=False,
)

validation_fixed_far_thresholds_df.to_csv(
    FIXED_FAR_THRESHOLD_PATH,
    index=False,
)

fixed_far_transfer_df.to_csv(
    FIXED_FAR_TRANSFER_PATH,
    index=False,
)

fixed_far_comparison_df.to_csv(
    FIXED_FAR_COMPARISON_PATH,
    index=False,
)

global_threshold_condition_df.to_csv(
    GLOBAL_CONDITION_PATH,
    index=False,
)

validation_condition_thresholds_df.to_csv(
    CONDITION_THRESHOLD_PATH,
    index=False,
)

condition_threshold_transfer_df.to_csv(
    CONDITION_TRANSFER_PATH,
    index=False,
)

threshold_diagnostic_summary_df.to_csv(
    THRESHOLD_DIAGNOSTIC_PATH,
    index=False,
)

finetuned_eer_validation = (
    validation_eer_thresholds_df[
        validation_eer_thresholds_df[
            "model"
        ] == "layer4_finetuned"
    ].iloc[0]
)

finetuned_eer_test = (
    eer_threshold_transfer_df[
        (
            eer_threshold_transfer_df[
                "model"
            ] == "layer4_finetuned"
        )
        & (
            eer_threshold_transfer_df[
                "experiment_split"
            ] == "official_test"
        )
    ].iloc[0]
)

finetuned_fixed_far = (
    validation_fixed_far_thresholds_df[
        validation_fixed_far_thresholds_df[
            "model"
        ] == "layer4_finetuned"
    ]
)

threshold_summary = {
    "threshold_selection_split": "validation",
    "models": list(
        score_columns.keys()
    ),
    "far_targets": FAR_TARGETS,
    "layer4_finetuned": {
        "validation_eer_threshold": float(
            finetuned_eer_validation[
                "threshold"
            ]
        ),
        "validation_eer_far": float(
            finetuned_eer_validation[
                "far"
            ]
        ),
        "validation_eer_frr": float(
            finetuned_eer_validation[
                "frr"
            ]
        ),
        "official_test_far_at_validation_eer_threshold": float(
            finetuned_eer_test[
                "far"
            ]
        ),
        "official_test_frr_at_validation_eer_threshold": float(
            finetuned_eer_test[
                "frr"
            ]
        ),
        "validation_tar_at_far_10_percent": float(
            finetuned_fixed_far.loc[
                finetuned_fixed_far[
                    "target_far"
                ] == 0.10,
                "tar",
            ].iloc[0]
        ),
        "validation_tar_at_far_1_percent": float(
            finetuned_fixed_far.loc[
                finetuned_fixed_far[
                    "target_far"
                ] == 0.01,
                "tar",
            ].iloc[0]
        ),
        "validation_tar_at_far_0_1_percent": float(
            finetuned_fixed_far.loc[
                finetuned_fixed_far[
                    "target_far"
                ] == 0.001,
                "tar",
            ].iloc[0]
        ),
        "condition_threshold_min": float(
            finetuned_thresholds.min()
        ),
        "condition_threshold_max": float(
            finetuned_thresholds.max()
        ),
        "condition_threshold_spread": float(
            finetuned_thresholds.max()
            - finetuned_thresholds.min()
        ),
    },
    "maximum_absolute_fixed_far_transfer_shift": float(
        fixed_far_comparison_df[
            "far_shift"
        ].abs().max()
    ),
    "maximum_absolute_fixed_tar_transfer_shift": float(
        fixed_far_comparison_df[
            "tar_shift"
        ].abs().max()
    ),
}

with SUMMARY_PATH.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        threshold_summary,
        file,
        indent=2,
    )

saved_paths = [
    EER_THRESHOLD_PATH,
    EER_TRANSFER_PATH,
    FIXED_FAR_THRESHOLD_PATH,
    FIXED_FAR_TRANSFER_PATH,
    FIXED_FAR_COMPARISON_PATH,
    GLOBAL_CONDITION_PATH,
    CONDITION_THRESHOLD_PATH,
    CONDITION_TRANSFER_PATH,
    THRESHOLD_DIAGNOSTIC_PATH,
    SUMMARY_PATH,
]

for path in saved_paths:
    print(
        path.name,
        ":",
        path.exists(),
    )

print()
print(
    "Fine-tuned validation EER threshold:",
    round(
        threshold_summary[
            "layer4_finetuned"
        ][
            "validation_eer_threshold"
        ],
        6,
    ),
)

print(
    "Fine-tuned validation TAR @ 1% FAR:",
    round(
        threshold_summary[
            "layer4_finetuned"
        ][
            "validation_tar_at_far_1_percent"
        ] * 100,
        4,
    ),
    "%",
)

print(
    "Fine-tuned condition-threshold spread:",
    round(
        threshold_summary[
            "layer4_finetuned"
        ][
            "condition_threshold_spread"
        ],
        6,
    ),
)

assert all(
    path.exists()
    for path in saved_paths
)

validation_eer_thresholds.csv : True
eer_threshold_transfer.csv : True
validation_fixed_far_thresholds.csv : True
fixed_far_threshold_transfer.csv : True
fixed_far_transfer_comparison.csv : True
global_threshold_condition_metrics.csv : True
validation_condition_eer_thresholds.csv : True
condition_threshold_transfer.csv : True
threshold_calibration_diagnostic.csv : True
verification_threshold_summary.json : True

Fine-tuned validation EER threshold: 0.925157
Fine-tuned validation TAR @ 1% FAR: 7.7381 %
Fine-tuned condition-threshold spread: 0.025299


## Threshold and operating-point conclusion

This experiment evaluated decision thresholds for writer verification using the fixed QUWI writer-disjoint protocol.

All thresholds were selected exclusively from the validation split and were then frozen before evaluation on the official-test split.

### Validation-derived EER threshold

For the layer-4 fine-tuned encoder, the global validation EER threshold was approximately 0.92516.

At this threshold:

- Validation FAR: 36.67%
- Validation FRR: 36.61%
- Validation TAR: 63.39%
- Official-test FAR: 33.73%
- Official-test FRR: 36.18%
- Official-test TAR: 63.82%

The validation-derived threshold therefore transferred reasonably consistently to the internal official-test writers.

### Fixed-FAR operating points

The fine-tuned encoder achieved the following validation performance:

- TAR 32.44% at approximately 10% FAR
- TAR 21.13% at approximately 5% FAR
- TAR 7.74% at approximately 1% FAR
- TAR 0.89% at approximately 0.1% FAR

Performance deteriorated sharply as the allowed false-accept rate became stricter. This demonstrates that the current supervised baseline is not adequate for reliable low-FAR writer verification.

Validation-derived fixed-FAR thresholds remained reasonably stable when transferred to the official-test writers. Across both evaluated models and all fixed-FAR operating points, the maximum absolute FAR shift was approximately 1.77 percentage points.

### Condition-dependent score calibration

A single global threshold produced substantially different error behavior across script and text conditions.

For the fine-tuned encoder using its global validation EER threshold, condition-level validation FAR ranged from approximately 24.68% to 53.54%, while FRR ranged from approximately 23.21% to 60.71%.

Condition-specific validation EER thresholds ranged from approximately 0.91029 to 0.93559, giving a threshold spread of approximately 0.02530.

For the fine-tuned encoder on validation data:

- Global-threshold mean absolute condition-level FAR-FRR gap: 0.2047
- Condition-specific-threshold mean absolute FAR-FRR gap: 0.0024

This large difference indicates that cosine similarity scores are not uniformly calibrated across the six verification conditions.

Condition-specific thresholds are used here only as a diagnostic tool. They are not proposed as the final deployment strategy, because a practical cross-script verifier should ideally support a common decision rule without requiring prior knowledge of the script or text condition.

### Research implication

The baseline representation contains transferable writer information, but two major limitations remain:

1. Verification performance is weak at strict false-accept operating points.
2. Similarity-score distributions change substantially across script and text conditions.

These observations motivate the next stages of the research: verification-oriented metric learning, self-supervised representation learning, and explicit analysis of script/content leakage and script invariance.

In [14]:
print(
    "Validation-derived fine-tuned EER threshold:",
    round(
        threshold_summary[
            "layer4_finetuned"
        ][
            "validation_eer_threshold"
        ],
        6,
    ),
)

print(
    "Fine-tuned validation TAR @ 10% FAR:",
    round(
        threshold_summary[
            "layer4_finetuned"
        ][
            "validation_tar_at_far_10_percent"
        ] * 100,
        4,
    ),
    "%",
)

print(
    "Fine-tuned validation TAR @ 1% FAR:",
    round(
        threshold_summary[
            "layer4_finetuned"
        ][
            "validation_tar_at_far_1_percent"
        ] * 100,
        4,
    ),
    "%",
)

print(
    "Fine-tuned validation TAR @ 0.1% FAR:",
    round(
        threshold_summary[
            "layer4_finetuned"
        ][
            "validation_tar_at_far_0_1_percent"
        ] * 100,
        4,
    ),
    "%",
)

print(
    "Condition threshold spread:",
    round(
        threshold_summary[
            "layer4_finetuned"
        ][
            "condition_threshold_spread"
        ],
        6,
    ),
)

print(
    "Maximum absolute FAR transfer shift:",
    round(
        threshold_summary[
            "maximum_absolute_fixed_far_transfer_shift"
        ] * 100,
        4,
    ),
    "percentage points",
)

print(
    "Notebook 10 complete:",
    True,
)

Validation-derived fine-tuned EER threshold: 0.925157
Fine-tuned validation TAR @ 10% FAR: 32.4405 %
Fine-tuned validation TAR @ 1% FAR: 7.7381 %
Fine-tuned validation TAR @ 0.1% FAR: 0.8929 %
Condition threshold spread: 0.025299
Maximum absolute FAR transfer shift: 1.7681 percentage points
Notebook 10 complete: True
